<a href="https://colab.research.google.com/github/Lianaulya/Practical-Statistics-for-Data-Scientists/blob/main/Chapter1_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Identitas Mahasiswa

| | |
|---|---|
| **Nama** | Muammar Yaldi Fadhil |
| **NIM** | 101032300019 |
| **Kelas** | TK-47-03 |
| **Mata Kuliah** | Machine Learning |

---

# Chapter 1 – Exploratory Data Analysis
**Book:** *Practical Statistics for Data Scientists* (O'Reilly)

This notebook reproduces the key concepts and code examples from Chapter 1:
- Estimates of location (mean, median, trimmed mean, weighted mean)
- Estimates of variability (std dev, MAD, IQR)
- Percentiles & Box Plots
- Frequency Tables & Histograms
- Density Plots
- Binary & Categorical Data
- Scatterplots & Correlation


In [ ]:
# Install required packages (run once)
# !pip install wquantiles

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import wquantiles

BASE_DATA = "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data"
print("Libraries loaded successfully!")

## 1. Estimates of Location

### Mean (Average)
The **mean** is the most basic measure of central location:

$$\bar{x} = \frac{\sum_{i=1}^{n} x_i}{n}$$

### Median
The **median** is the middle value when data is sorted – it is robust to outliers.

### Trimmed Mean
A **trimmed mean** removes a fixed proportion of the smallest and largest values before computing the average.
This reduces the effect of extreme outliers.

### Weighted Mean
$$\bar{x}_w = \frac{\sum_{i=1}^{n} w_i x_i}{\sum_{i=1}^{n} w_i}$$

In [ ]:
# Table 1-2: State population and murder rate data
state = pd.read_csv(f"{BASE_DATA}/state.csv")
print(state.head(8))

In [ ]:
# Mean, trimmed mean (10%), weighted mean
mean_pop = state["Population"].mean()
trimmed_mean = stats.trim_mean(state["Population"], 0.1)
weighted_mean = np.average(state["Murder.Rate"], weights=state["Population"])

print(f"Mean Population:         {mean_pop:,.0f}")
print(f"Trimmed Mean (10%):      {trimmed_mean:,.0f}")
print(f"Weighted Mean (Murder):  {weighted_mean:.4f}")

# Median
print(f"Median Population:       {state['Population'].median():,.0f}")

## 2. Estimates of Variability

### Variance and Standard Deviation
$$s = \sqrt{\frac{\sum_{i=1}^{n}(x_i - \bar{x})^2}{n-1}}$$

### Mean Absolute Deviation (MAD)
$$MAD = \frac{\sum_{i=1}^{n}|x_i - \bar{x}|}{n}$$

### Median Absolute Deviation from Median
$$Median\;MAD = Median(|x_i - m|)$$ where $m$ is the median.\n\n### Percentiles & IQR
$$IQR = Q_3 - Q_1$$ (75th percentile − 25th percentile)

In [ ]:
# Variability estimates
pop = state["Population"]

print(f"Standard Deviation:   {pop.std():,.0f}")
print(f"IQR:                  {pop.quantile(0.75) - pop.quantile(0.25):,.0f}")
print(f"MAD:                  {(pop - pop.mean()).abs().mean():,.0f}")
print(f"Median Abs Dev:       {(pop - pop.median()).abs().median():,.0f}")

## 3. Percentiles and Box Plots

Percentiles partition the data into 100 equal parts. The **box plot** visualizes the five-number summary:
- Minimum (or 5th percentile)
- Q1 (25th percentile)
- Median (50th percentile)
- Q3 (75th percentile)
- Maximum (or 95th percentile)


In [ ]:
# Percentiles of murder rate
percentiles = [5, 25, 50, 75, 95]
print("Percentiles of Murder Rate:")
for p in percentiles:
    print(f"  P{p:2d}: {state['Murder.Rate'].quantile(p/100):.2f}")

In [ ]:
# Box plot of population by region
fig, ax = plt.subplots(figsize=(8, 4))
state.boxplot("Population", ax=ax)
ax.set_title("Population Distribution Across US States")
ax.set_ylabel("Population")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
plt.tight_layout()
plt.show()

## 4. Frequency Tables and Histograms

A **frequency table** bins continuous data into discrete intervals and counts how many observations fall in each bin.
A **histogram** is the graphical version of a frequency table.

In [ ]:
# Frequency table of murder rates
bins = [1, 2.5, 4, 5.5, 7, 9, 11]
freq, edges = np.histogram(state["Murder.Rate"], bins=bins)
freq_df = pd.DataFrame({"BinStart": edges[:-1], "BinEnd": edges[1:], "Count": freq})
print("Murder Rate Frequency Table:")
print(freq_df.to_string(index=False))

In [ ]:
# Histogram of murder rate
fig, ax = plt.subplots(figsize=(8, 4))
state["Murder.Rate"].hist(bins=bins, ax=ax, edgecolor="black", color="steelblue")
ax.set_title("Histogram of State Murder Rates")
ax.set_xlabel("Murder Rate (per 100,000)")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# DFW airline delays — bar chart
dfw = pd.read_csv(f"{BASE_DATA}/dfw_airline.csv")
print(dfw.head())

fig, ax = plt.subplots(figsize=(9, 4))
dfw.transpose().plot(kind="bar", ax=ax, legend=False, color="steelblue", edgecolor="black")
ax.set_title("DFW Airport – % Delayed Flights by Airline")
ax.set_xlabel("Airline")
ax.set_ylabel("% of Delays")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Density Plots and Estimates

A **density plot** (KDE – Kernel Density Estimate) is a smoothed version of a histogram.
It shows the *shape* of the distribution without being sensitive to bin choice.

KDE uses a smoothing kernel $K$ with bandwidth $h$:

$$\hat{f}(x) = \frac{1}{nh} \sum_{i=1}^{n} K\!\left(\frac{x - x_i}{h}\right)$$

In [ ]:
# KDE / density of murder rate
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram + KDE
state["Murder.Rate"].plot.hist(bins=10, density=True, ax=axes[0], color="lightsteelblue", edgecolor="black")
state["Murder.Rate"].plot.kde(ax=axes[0], color="navy")
axes[0].set_title("Murder Rate – Histogram + KDE")
axes[0].set_xlabel("Murder Rate")

# Pure KDE
state["Murder.Rate"].plot.kde(ax=axes[1], color="steelblue", linewidth=2)
axes[1].set_title("Murder Rate – Kernel Density Estimate")
axes[1].set_xlabel("Murder Rate")

plt.tight_layout()
plt.show()

## 6. Exploring Binary and Categorical Data

For binary (0/1) or categorical data we summarize using:
- **Proportions / mode** for categorical
- **Expected value**: $E(X) = \sum_{i} x_i \cdot P(x_i)$
- **Bar charts** for visualization

In [ ]:
# Lending Club loans — categorical counts
lc_loans = pd.read_csv(f"{BASE_DATA}/lc_loans.csv")
print(lc_loans["status"].value_counts())
print("\nProportions:")
print(lc_loans["status"].value_counts(normalize=True).round(3))

In [ ]:
# Stacked bar chart by status and term
pivot = lc_loans.pivot_table(index="grade", columns="status", values="amount",
                              aggfunc="sum", fill_value=0)
fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind="bar", stacked=True, ax=ax, colormap="tab10")
ax.set_title("Loan Amount by Grade and Status")
ax.set_xlabel("Grade")
ax.set_ylabel("Total Amount ($)")
ax.legend(title="Status", bbox_to_anchor=(1.01, 1))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Correlation

The **Pearson correlation coefficient** measures linear association:

$$r = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{(n-1)\,s_x s_y}$$

- $r = 1$: perfect positive linear correlation
- $r = -1$: perfect negative linear correlation
- $r = 0$: no linear correlation


In [ ]:
# S&P 500 sector correlations
sp500_sym = pd.read_csv(f"{BASE_DATA}/sp500_sectors.csv")
sp500_px = pd.read_csv(f"{BASE_DATA}/sp500_data.csv.gz", index_col=0)
print("S&P500 data shape:", sp500_px.shape)
print(sp500_sym.head())

In [ ]:
# Compute telecom sector correlations
telecom = sp500_sym[sp500_sym["sector"] == "telecommunications_services"]["symbol"]
telecom_px = sp500_px[telecom]
corr_matrix = telecom_px.corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, square=True)
ax.set_title("Telecom Sector – Return Correlations")
plt.tight_layout()
plt.show()

## 8. Scatterplots

A **scatterplot** shows the relationship between two continuous variables.
We can use alpha blending and hexbin for large datasets.

In [ ]:
# King County, WA – Tax assessed value vs. finished sqft
kc_tax = pd.read_csv(f"{BASE_DATA}/kc_tax.csv.gz")
kc_tax = kc_tax.dropna()
kc_tax = kc_tax[(kc_tax["TaxAssessedValue"] < 750000) &
                (kc_tax["SqFtTotLiving"] < 3500) &
                (kc_tax["SqFtTotLiving"] > 300)]
print(f"KC Tax data: {kc_tax.shape[0]:,} rows after filtering")

In [ ]:
# Hexbin scatterplot for large dataset
fig, ax = plt.subplots(figsize=(8, 5))
kc_tax.plot.hexbin("SqFtTotLiving", "TaxAssessedValue",
                   gridsize=30, sharex=False, ax=ax,
                   colormap="Blues", edgecolors="none")
ax.set_title("KC Tax Value vs. Living Area")
ax.set_xlabel("Square Feet (Total Living)")
ax.set_ylabel("Tax Assessed Value ($)")
plt.tight_layout()
plt.show()

## Summary

| Concept | Key Formula / Tool |
|---------|-------------------|
| Mean | $\bar{x} = \frac{\sum x_i}{n}$ |
| Trimmed Mean | Remove top/bottom $p\%$, then average |
| Weighted Mean | $\frac{\sum w_i x_i}{\sum w_i}$ |
| Std Deviation | $s = \sqrt{\frac{\sum(x_i-\bar{x})^2}{n-1}}$ |
| IQR | $Q_3 - Q_1$ |
| Pearson Corr | $r \in [-1, 1]$ |
